In [0]:
USE CATALOG meridian_t02;

-- =========================================================================
-- 1. ROW FILTER FUNCTION: Evaluates user_country entitlements
-- =========================================================================
CREATE OR REPLACE FUNCTION gov.fn_country_row_filter(country_val STRING)
RETURNS BOOLEAN
LANGUAGE SQL
DETERMINISTIC
READS SQL DATA
COMMENT 'Row filter: returns TRUE if user is in t02_admins or explicitly mapped to the country'
RETURN 
    -- Admins see all rows unconditionally
    is_account_group_member('t02_admins')
    OR
    -- Check if user or user group is listed in the entitlement table
    EXISTS (
        SELECT 1 
        FROM meridian_t02.gov.entitlements e
        WHERE (e.principal = current_user() OR is_account_group_member(e.principal))
          AND e.allowed_country = country_val
    );

-- =========================================================================
-- 2. COLUMN MASK FUNCTION: User Email (Partial Masking)
-- =========================================================================
CREATE OR REPLACE FUNCTION gov.fn_mask_email(email_val STRING)
RETURNS STRING
LANGUAGE SQL
DETERMINISTIC
CONTAINS SQL
COMMENT 'Masks email to show first 2 characters and domain unless in t02_pii_readers'
RETURN 
    CASE 
        WHEN is_account_group_member('t02_pii_readers') THEN email_val
        WHEN email_val IS NULL THEN NULL
        ELSE REGEXP_REPLACE(email_val, '(?<=^.{2}).*?(?=@)', '****')
    END;

-- =========================================================================
-- 3. COLUMN MASK FUNCTION: Review Free-Text Comment (Full Redaction)
-- =========================================================================
CREATE OR REPLACE FUNCTION gov.fn_mask_comment(comment_val STRING)
RETURNS STRING
LANGUAGE SQL
DETERMINISTIC
CONTAINS SQL
COMMENT 'Redacts review comments to [REDACTED] unless caller belongs to t02_pii_readers'
RETURN 
    CASE 
        WHEN is_account_group_member('t02_pii_readers') THEN comment_val
        WHEN comment_val IS NULL THEN NULL
        ELSE '[REDACTED]'
    END;